In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Distributions and the Central-Limit Theorem

In class, we did an experiment where each of you rolled a (presumably fair) die and counted the number of rolls until you got a six.
We didn't say at the time, but the number of rolls until you get a 6 is a **random variable**, call it $X$. That is, $X$ represents the observation of a data collection procedure, and its outcome was, well, random.
Technically speaking, a random variable is neither random, nor a variable.
Instead, a random variable is described according to the distribution its outcomes obey.

For the dice-rolling experiment, the random variable follows a [geometric distribution](https://en.wikipedia.org/wiki/Geometric_distribution), which describes the number of trials required to get a success, where the probability of success is $p$.
For rolling a fair-six-sided, die, $p = 1/6$.
We express the random variable $X$ according to its distribution,
\begin{equation*}
X\sim G(p) \ \ \Rightarrow \ \ X\sim G(1/6)
\end{equation*}
So here we see that the random variable $X$ is more generally described by the distribution it follows. You all observed many outcomes of that random variable.

## A sidebar into probability mass functions
Technically speaking the Geometric distribution describes a _discrete_ random variable. That is, a random variable whose outcomes can be enumerated (e.g., 1 roll, 7 rolls, 62 rolls).
The probability of individual outcomes can be described via a _probability mass function_, $f(x)$. For the geometric distribution, we know (with some careful thinking) its probability mass function,
\begin{equation}\label{E:geoPDF}
    \Pr(X=x) = (1-p)^{x-1} p, \ \ \text{where } x = 1, 2, 3, 4, \dots.
\end{equation}
In other words, for a random variable $X\sim G(p)$ its mass function is $f(x) = (1-p)^{x-1} p$.
For the dice example, we can calculate the probability that it takes 3 rolls to achieve a 6,
\begin{equation*}
    \Pr(X=3) = (1-1/6)^{3-1}\cdot(1/6)\approx 0.1157
\end{equation*}
You might interpret this probability as saying if you oberved 100 dice-rolling-expirements, the outcome would be three somewhere between 11 and 12 times.
Of course in reality, you might not get exactly 11 or 12 (or 11.57), but we would expect something close to these values.

First, let's repeat the dice rolling experiment, but ask numpy to do the rolling for us using `np.random.geometric()`

In [ ]:
sims_per_trial = 100
number_trials  = 1200

# Fair die
prob_success   = 1/6

# Generate numpy array with (number of rows) x (number of columns) = (number_trials) x (sims_per_trial)
trials = np.random.geometric(prob_success, size=(number_trials, sims_per_trial))

In [ ]:
# compute the mean for each trial
trials_mean = np.mean( trials, axis=1)

We can visualize the number of rolls for _all_ simulations.
By this, we borrow the "flatten" convention from `numpy` and treat every entry in our simulated array as its own dice-rolling-experiment (which it is).
For the flattened array, we can use `np.histogram()` to bin our data and count the frequency of each outcome.

In [ ]:
# plot a histogram for all (flattened) trials

hist_all, bins_all = np.histogram( trials, bins = range(1, np.max(trials)+1, 1));

plt.hist(bins_all[:-1], bins_all, weights = hist_all, density=False)
plt.xticks(range(1, np.max(trials)+2,2), rotation=90);
plt.xlabel('Number of rolls until a 6');
plt.ylabel('Frequency of each count');
plt.title('All dice rolling experiments');

The above histogram indictes a general shape or _distribution_ of the data.
It certainly appears to follow some trend or shape.
The histogram forms what we call an `empircal` distribution, or, a distribution that is derived from data collection or simulation.
It follows such a 'nice' pattern because we have done so many simulations.
Alternatively, we could plot the histogram for a single trial, and see that some shape seems to underly the distribution, but the empirical distribution lacks a "smooth" shape.

In [ ]:
# We can plot a histogram for a single trial. It looks less 'nice', but still seems to follow a trend or shape

trial_number = 8
hist_trial, bins_trial = np.histogram( trials[trial_number,:], bins = range(1, np.max(trials[trial_number,:])+1, 1));

plt.hist(bins_trial[:-1], bins_trial, weights = hist_trial, density=False)
plt.xticks(bins_trial);
plt.xlabel('Number of rolls until a 6');
plt.ylabel('Frequency of each count');
plt.title('Single trial dice rolling experiment');

It turns out that the probability mass function from Eq. \eqref{E:geoPDF} can be superimposed on our histogram.
We will plot the pmf as a curve, even though it is only evaluated at integer values.
I only like the curve because we can observe its closeness to the histogram. But remember, the pmf is technically only evaluated at the discrete outcomes (and not all of the points in between the outcomes, as illustrated by  a curve).
But we have to scale the heights of the histogram so that they can represent probabilities, and not raw counts.
We accomplish this by setting `density=True`, which normalizes the heights so that the total binned area _integrates_ to one.

To visualize the similarity between the emperical and theoretical distribution, we observe that the curve (approximately) passes through the upper-left corner of each histogram bin.


In [ ]:
# How does the histogram compare to the expected pdf?
# For each simulation X \sim G(1/6)
# The expected value is 1/p
# The variance is (1-p)/p^2
#
# Each trial represents a random sampling of t 10 observations of the geometric variable
# The average for each trial is distributed with mean mu=1/p and variance sigma^2 = (1-p)/p^2 / n

# First does the overall sampling reproduce the expected distribution
pdf_geo = lambda x, p: (1-p)**(x-1) * p

xx_geo = np.linspace(0,50,100)
yy_geo = pdf_geo( xx_geo, prob_success)

plt.hist(bins_all[:-1], bins_all, weights = hist_all, density=True, label='Geometric empirical distribution');
plt.plot(xx_geo, yy_geo, label='Geometric pmf')
plt.legend()
plt.xlabel('Number of rolls until a 6');
plt.ylabel('Frequency of each count');
plt.title('Empirical vs. Theoretical distributions');

## A sidebar into probability density functions

For continuous random variables (random variables whose outcomes cannot be enumerated), we still need a notion of "probability mass".
But this gets trickier because the probability of and specific event occurring is zero. That is, $\Pr(X=x) = 0$. Why is this? That's for you to ponder.
Instead, for continuous random variables we think of probabilities associated with ranges of outcomes. In symbols, this means we want to consider $\Pr(a\leq X \leq b)$.

So, why density? Well, consider the interval of outcomes $[a,b]$---this represents a length.
I want to determine the "mass" of probability above this length.
For this, I recall the relation:
\begin{equation*}
\text{density} = \frac{\text{mass}}{\text{volume}}.
\end{equation*}
In fact, my length can be thought of as a 1-dimensional volume, or a linear volume.
So, if I can quantify a "linear density" for my probability, I can find the probabilitymass as $\text{density}\times\text{volume}$.

This is where probability density functions come in. They are functions that describe the _probability density_ at outcomes $x$, rather than the probabilities of individual outcomes themselves. Remember, the probability of individual outcomes is zero, so a function that describes individual probabilities is not very interesting.

The most well-known continuous probability density function (pdf) is for a random variable that follows a normal distribution with mean $\mu$ and variance $\sigma^2$; $X\sim N(0,\sigma^2)$. The pdf is given by
\begin{equation}\label{E:normPDF}
f(x) = \frac{1}{\sqrt{2\pi\sigma^2}}e^{-\frac{(x-u)^2}{2\sigma^2}}
\end{equation}

There are a lot of weird pieces in this formula, but perhaps you remember that $g(x) = e^{-x^2}$ is a little bump function centered at $x=0$ with $g(0) = 1$.
The normal pdf shifts this graph to instead be centered at $x=\mu$, scales the bump horizontally to capture the spread of the data, and scales the bump vertically so that the total probability mass adds to 1.

For completeness, we state that probabilities are calculated via definite integrals with the pdf,
\begin{equation*}
    \Pr(a\leq X \leq b) = \int_a^b f(x)\,\mathrm{d}x
\end{equation*}

## The Central Limit Theorem
It turns out the normal distribution plays a big roll in all types of data, even those which don't follow a normal distribution themselves.

**The Central Limit Theorem** Consider a sample of random variables $X_1, X_2, \dots, X_N$ such that:
* The random variables are independent; the outcome of one does not affect the outcome of any other.
* The random variable follow an identical distribution with mean $\mu$ and variance $\sigma^2$
Consider the mean of the random variables,
\begin{equation*}
    \bar{X}_{N} = \frac{X_1 + X_2 + \dots + X_N}{N}.
\end{equation*}

As $N\to\infty$, the sample mean has distribution $\mu$ and variance $\sigma^2 / N$. Or, in symbols: $\bar{X}_N \sim N(\mu,\sigma^2/N)$ as $N\to\infty$.



We note that in the statement of the theorem, the mean $\mu$ and variance $\sigma^2$ are _known_. In practice this is rarely the case. But for our dice rolling experiment with $X\sim G(p)$ (with $p=1/6$), we have formulas for the mean and variance:
\begin{align*}
    \mu & = \frac{1}{p}  &\implies \mu &= \frac{1}{6}\\
    \sigma^2 & = \frac{1-p}{p^2}  &\implies \sigma^2  &= 30
\end{align*}

Note that these are the mean and variance of the _geometric_ distribution.
We would not expect these to be the mean and variance of the _average_ of geometric simulations.

We visualize the central limit theorem by finding the mean for each geometric simulation, and plotting a _histogram of the means_. We again have to set `density=True`.
We can superimpose the normal pdf (with properly adjusted mean and variance)

In [ ]:
# Mean and variance from the geometric distribution
mu_geo = 1/prob_success
sigma_sq_geo = (1-prob_success) / (prob_success**2)


# adjust for the central limit theorem
mu_clt = mu_geo
sigma_sq_clt = sigma_sq_geo / sims_per_trial

# create lambda expression for the normal pdf
pdf_mean = lambda x, mu, sigma_sq: 1 / np.sqrt(2*np.pi*sigma_sq) * np.exp( -(x-mu)**2 / (2*sigma_sq) )

# creat an input of points to evaluate the pdf
xx_mean = np.linspace(0,12,500)

# evaluate the pdf with the CLT mu and sigma
yy_mean = pdf_mean(xx_mean, mu_clt, sigma_sq_clt)

hist_mean, bins_mean = np.histogram(trials_mean, bins=20)

plt.hist(bins_mean[:-1], bins_mean, weights = hist_mean, density=True, label='Empirical distribution')

plt.plot(xx_mean, yy_mean, label = 'normal pdf')
plt.legend()
plt.xlabel('Average Number of rolls until a 6');
plt.ylabel('Frequency of each count');
plt.title('Visualizing the CLT');

Note that we have two variables that we play with here: `sims_per_trial` and `number_trials`.
The way we have formulated the code, we are indicating that `sims_per_trial` corresponds to the $N$ in the central limit theorem.
As `sims_per_trial` gets large, the variance of $\bar{X}_N$ gets smaller, and we converge to a really narrow normal distribution.
As `number_trials` gets large, our empirical distribution will look more and more like the theoretical distribution.

### An exercise:
Find the percentage of trial means that lie:
* within 1 standard deviation of the theoretical trial mean
* within 2 standard deviations of the theoretical trial mean
* within 3 standard deviations of the theoretical trial mean

In [ ]:
def count_events(data, lower_bound, upper_bound):
    # find which data points x, satisfy lower_bound <= x <= b
    
    # Use booleans
    idx_bool = (data >= lower_bound) & (data <= upper_bound)
    
    # add booleans (which is adding 1's and 0's)
    total_events = np.sum(idx_bool)
    return total_events, idx_bool


multipliers = [1.0, 2.0, 3.0]

for j in range( len(multipliers) ):
    M = multipliers[j]
    count,_ = count_events( trials_mean, mu_clt - M * np.sqrt(sigma_sq_clt), mu_clt + M * np.sqrt(sigma_sq_clt))
    perc = count / number_trials * 100
    print('There are %5.2f %% of trial means within %2.2f standard deviations of the theoretical mean' %(perc, M))

You might recall that we expect these numbers to be $68.27\,\%$, $95.45\,\%$, and $99.73\,\%$, respectively.
Due to randomness with the finite sample size, we are unlikely to get these numbers exactly.

An interesting note: even in simulation, we did get _rare_ trial means that were a (relatively) large distance away from the expected sample mean. Hmmmmm....